# 701장 실행 + OCR 원문 저장 (예선 최종본 그대로)

예선 최종본(`main`)의 OCR + Parser를 **코드 수정 없이** 701장에 돌리고,
각 이미지에서 **OCR이 읽은 글자(4단계 전부)**를 저장합니다.
저장된 OCR 원문이 있으면 이후 Parser를 고칠 때 OCR을 다시 돌리지 않고 몇 초 만에 재평가할 수 있습니다.

## 사용법
`itda_OCR` 폴더에서 설치(`.venv`, `requirements.txt`)와 가중치 다운로드(`scripts\\download_weights.py`)를 끝낸 뒤,
이 노트북을 열고 **커널을 `.venv`로 선택 → 메뉴에서 "모두 실행(Run All)"**. 수정할 곳은 없습니다.

- 1번 셀: 필요한 파일·폴더가 없으면 무엇이 없는지 한글로 알려주고 멈춥니다.
- 5번 셀: 본 실행 전에 2장을 예선 최종본 `predict_image`로 직접 돌려 결과가 같은지 자동 확인합니다. 다르면 멈춥니다.
- 6번 셀: 701장 본 실행. 이미지당 4단계를 모두 실행하므로 **약 60~75분**. 10장마다 남은 시간을 보여줍니다.
- 중간에 멈추거나 PC가 꺼져도, 다시 "모두 실행"하면 **끝난 이미지는 건너뛰고 이어서** 진행합니다.

## 결과 (OUTPUT_DIR 안)
| 파일 | 내용 |
|---|---|
| `ocr_dump.jsonl` | 이미지별 4단계 OCR 원문(글자·신뢰도·위치) ← **가장 중요, Claude에게 전달** |
| `predictions.csv` | 예선 최종본과 동일한 방식의 최종 예측 |
| `evaluation.csv` | 정답 비교 + 오답의 OCR/Parser 자동 1차 분류 |
| `summary.json` | 정확도 요약, 실행 환경 |

In [ ]:
# ===== CONFIG (기본값 그대로 두면 됩니다) =====
from pathlib import Path

REPO_DIR   = Path(r"C:\Users\user\itda_OCR")                  # 예선 최종본 파일을 풀어 넣은 폴더
IMAGE_DIR  = Path(r"C:\Users\user\itda_OCR\상품사진입니다")      # 전체 3,352장 폴더
LABELS_CSV = Path(r"C:\Users\user\itda_OCR\labels_701.csv")    # 701장 라벨 (이 위치에 저장)
OUTPUT_DIR = Path(r"C:\Users\user\itda_OCR\run701_output")     # 결과 저장 폴더 (없으면 자동 생성)

CPU_THREADS    = 4       # PC 코어 수에 맞게 (예선 제출 설정은 2 프로세스 x 2 스레드)
RUN_ALL_STAGES = True    # True: 4단계 전부 실행해 OCR 원문 저장(권장) / False: 예선처럼 후보 나오면 중단(빠름)
LIMIT          = None    # 숫자를 넣으면 그 장수만 실행 (예: 5). None = 701장 전부
# ====================================

In [ ]:
# 1) 예선 최종본 코드 불러오기 (코드는 수정하지 않고 import만 함)
import sys, os, json, csv, time, subprocess, platform, re
for need in ["ocr_pipeline.py", "date_parser", r"weights/paddleocr/PP-OCRv6_medium_det", r"weights/paddleocr/korean_PP-OCRv5_mobile_rec"]:
    if not (REPO_DIR / need).exists():
        raise FileNotFoundError(f"REPO_DIR에 {need} 이(가) 없습니다: {REPO_DIR}\n"
                                "예선 최종본(main) 폴더를 지정했는지, download_weights.sh를 실행했는지 확인하세요.")
for p, name in [(IMAGE_DIR, "IMAGE_DIR"), (LABELS_CSV, "LABELS_CSV")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} 경로가 없습니다: {p}")
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

import numpy as np
from PIL import Image
import ocr_pipeline as op
from date_parser import parse_expiration_date

def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"unavailable: {e}"

version_file = REPO_DIR / "MAIN_VERSION.txt"
ENV = {
    "main_version_file": version_file.read_text(encoding="utf-8").strip() if version_file.exists() else "없음",
    "repo_commit": git("rev-parse", "HEAD"),
    "repo_branch": git("rev-parse", "--abbrev-ref", "HEAD"),
    "repo_dirty_files": git("status", "--porcelain", "--", "ocr_pipeline.py", "date_parser"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu_threads": CPU_THREADS,
    "run_all_stages": RUN_ALL_STAGES,
}
print(json.dumps(ENV, ensure_ascii=False, indent=1))
if version_file.exists():
    print("예선 최종본 버전:", ENV["main_version_file"].splitlines()[0])
elif ENV["repo_branch"] != "main":
    print("주의: 예선 최종본 버전을 확인할 수 없습니다 (MAIN_VERSION.txt 없음, git main 아님).")

In [ ]:
# 2) 라벨(정답) 불러오기: labels_701.csv (image_id,year,month,day,final_date,truth_source,label_sources)
#    truth_source = approved(검수 확정 62장) / candidate(두 팀 라벨 일치 639장)
with open(LABELS_CSV, encoding="utf-8-sig", newline="") as f:
    label_rows = list(csv.DictReader(f))

need = {"image_id", "year", "month", "day", "final_date", "truth_source", "label_sources"}
if not need <= set(label_rows[0]):
    raise ValueError(f"LABELS_CSV에 필요한 열이 없습니다: {sorted(need - set(label_rows[0]))}. labels_701.csv를 지정하세요.")

truth = {}
for r in label_rows:
    truth[str(int(r["image_id"]))] = {
        "source": r["truth_source"], "year": r["year"].strip(), "month": r["month"].strip(),
        "day": r["day"].strip(), "final_date": r["final_date"].strip(), "label_sources": r["label_sources"],
    }

print("라벨 행 수:", len(truth))
print("approved:", sum(t["source"] == "approved" for t in truth.values()),
      "/ candidate:", sum(t["source"] == "candidate" for t in truth.values()))
empty = [i for i, t in truth.items() if not t["final_date"]]
print("정답이 비어 있는 행:", empty or "없음")

In [ ]:
# 3) 이미지 파일 찾기 (파일명 앞자리 0 무시: 000007.jpg → 7)
exts = {".jpg", ".jpeg", ".png"}
files = {}
for p in IMAGE_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in exts and p.stem.isdigit():
        files.setdefault(str(int(p.stem)), []).append(p)

missing = sorted((i for i in truth if i not in files), key=int)
duplicated = {i: v for i, v in files.items() if i in truth and len(v) > 1}
print(f"701장 중 찾은 이미지: {len(truth) - len(missing)}장")
if missing:
    print(f"못 찾은 image_id {len(missing)}개 (앞 20개):", missing[:20])
    (Path(OUTPUT_DIR) / "missing_images.txt").parent.mkdir(parents=True, exist_ok=True)
    (Path(OUTPUT_DIR) / "missing_images.txt").write_text("\n".join(missing), encoding="utf-8")
    print("전체 목록:", Path(OUTPUT_DIR) / "missing_images.txt")
else:
    print("못 찾은 이미지: 없음")
if duplicated:
    print("같은 번호 파일이 여러 개 (첫 번째 사용):", {k: [str(x) for x in v] for k, v in duplicated.items()})

targets = sorted((i for i in truth if i in files), key=int)
if LIMIT:
    targets = targets[:LIMIT]
print("이번 실행 대상:", len(targets), "장")

In [ ]:
# 4) OCR 엔진 준비 (예선 최종본 설정 그대로)
engine = op.initialize_engine(enable_mkldnn=True, cpu_threads=CPU_THREADS, recognition_batch_size=6)
print("OCR 엔진 준비 완료")

In [ ]:
# 5) 사전 검증: 이 노트북의 실행 방식이 예선 최종본 predict_image와 같은 결과를 내는지 2장으로 확인
#    - 최종 예측은 predict_image와 같은 규칙(후보가 처음 나온 단계 채택, 없으면 첫 단계 결과)
#    - RUN_ALL_STAGES=True면 후보가 나와도 나머지 단계까지 실행해 OCR 원문을 모두 저장
def run_one(path):
    rgb = op.decode_image(path)
    base = op.resize_image(rgb, 512)
    stages = (
        ("original_512", lambda: base, 512),
        ("rotation_270", lambda: np.asarray(Image.fromarray(base).rotate(270, expand=True)), 512),
        ("highres_1024", lambda: op.resize_image(rgb, 1024), 1024),
        ("clahe", lambda: op.apply_clahe(base), 512),
    )
    records, original, chosen = [], None, None
    for method, prepare, side in stages:
        t0 = time.perf_counter()
        detections = op.paddle_to_common(engine.predict(
            prepare(), text_det_limit_side_len=side,
            text_det_limit_type="max", text_det_box_thresh=0.7,
        ))
        seconds = time.perf_counter() - t0
        prediction = parse_expiration_date(detections)
        candidate = op.has_candidate(detections)
        records.append({"stage": method, "seconds": round(seconds, 3), "has_candidate": candidate,
                        "prediction": prediction, "detections": detections})
        if original is None:
            original = prediction
        if candidate and chosen is None:
            chosen = (prediction, method)
            if not RUN_ALL_STAGES:
                break
    if chosen is None:
        chosen = (original, "original_no_candidate")
    return chosen, records

for image_id in targets[:2]:
    path = files[image_id][0]
    expected = op.predict_image(engine, path)
    got, _ = run_one(path)
    if tuple(expected) != tuple(got):
        raise RuntimeError(f"{path.name}: predict_image={expected} / 노트북={got} → 결과가 다릅니다. 이 화면을 Claude에게 보내주세요.")
    print(f"{path.name}: 예선 최종본과 일치 {got}")
print("사전 검증 통과")

In [ ]:
# 6) 본 실행 (이어하기 지원: 성공한 이미지는 건너뛰고, 실패했던 이미지는 다시 시도)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DUMP = OUTPUT_DIR / "ocr_dump.jsonl"

def load_dump():
    records = {}
    if DUMP.exists():
        with open(DUMP, encoding="utf-8") as f:
            for line in f:
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue  # 강제 종료로 잘린 줄은 무시
                if "error" not in r or r["image_id"] not in records:
                    records[r["image_id"]] = r
    return records

done = {i for i, r in load_dump().items() if "error" not in r}
todo = [i for i in targets if i not in done]
print(f"이미 완료 {len(done & set(targets))}장, 이번에 실행 {len(todo)}장")

# 강제 종료로 마지막 줄이 잘려 있으면 줄바꿈을 넣어 다음 기록과 붙지 않게 함
if DUMP.exists() and DUMP.stat().st_size:
    with open(DUMP, "rb") as f:
        f.seek(-1, os.SEEK_END)
        needs_newline = f.read(1) != b"\n"
    if needs_newline:
        with open(DUMP, "a", encoding="utf-8") as f:
            f.write("\n")

started = time.perf_counter()
with open(DUMP, "a", encoding="utf-8") as out:
    for n, image_id in enumerate(todo, 1):
        path = files[image_id][0]
        try:
            (prediction, method), stages = run_one(path)
            rec = {"image_id": image_id, "file": path.name, "method": method,
                   "prediction": prediction, "stages": stages}
        except Exception as e:  # 한 장 실패해도 전체는 계속
            rec = {"image_id": image_id, "file": path.name, "error": repr(e)}
            print(f"  {path.name} 실패: {e!r}")
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        out.flush()
        if n % 10 == 0 or n == len(todo):
            el = time.perf_counter() - started
            eta = el / n * (len(todo) - n)
            print(f"[{n}/{len(todo)}] 경과 {el/60:.1f}분, 남은 예상 {eta/60:.1f}분", flush=True)
print("실행 완료")

In [ ]:
# 7) 채점 + 오답 1차 분류
#    OCR 원문(4단계 중 어디든)에 정답 날짜 숫자가 한 줄 안에 모두 보이면 → "Parser 의심"
#    안 보이면 → "OCR 의심"   (자동 1차 분류일 뿐, 최종 판단은 사람이 확인)
dumped = {i: r for i, r in load_dump().items() if i in truth}

def digits_visible(t, stages):
    need = []
    if t["year"] != "NONE":
        need.append({t["year"], t["year"][2:]})
    if t["month"] != "NONE":
        need.append({t["month"], str(int(t["month"]))})
    if t["day"] != "NONE":
        need.append({t["day"], str(int(t["day"]))})
    if not need:
        return False
    for s in stages:
        for det in s["detections"]:
            nums = set(re.findall(r"\d+", det["text"]))
            for n in list(nums):
                if len(n) == 8:  # 20210915 처럼 붙은 숫자
                    nums |= {n[:4], n[4:6], n[6:]}
            if all(opts & nums for opts in need):
                return True
    return False

pred_rows, eval_rows = [], []
for image_id in sorted(dumped, key=int):
    r, t = dumped[image_id], truth[image_id]
    if "error" in r:
        eval_rows.append({"image_id": image_id, "error": r["error"]})
        continue
    p = r["prediction"]
    pred_rows.append({"image_id": Path(r["file"]).stem, **p})
    ok = p["final_date"] == t["final_date"]
    if ok:
        kind = ""
    elif digits_visible(t, r["stages"]):
        kind = "Parser 의심"
    else:
        kind = "OCR 의심"
    chosen = next((s for s in r["stages"] if s["stage"] == r["method"]), r["stages"][0])
    eval_rows.append({
        "image_id": image_id, "truth_source": t["source"], "label_sources": t["label_sources"],
        "true_final_date": t["final_date"], "pred_final_date": p["final_date"],
        "correct": ok, "method": r["method"], "first_guess": kind,
        "chosen_stage_text": " | ".join(d["text"] for d in chosen["detections"]),
        "error": "",
    })

with open(OUTPUT_DIR / "predictions.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["image_id", "year", "month", "day", "final_date"])
    w.writeheader(); w.writerows(pred_rows)
fields = ["image_id", "truth_source", "label_sources", "true_final_date", "pred_final_date",
          "correct", "method", "first_guess", "chosen_stage_text", "error"]
with open(OUTPUT_DIR / "evaluation.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields, restval="")
    w.writeheader(); w.writerows(eval_rows)
print("저장:", OUTPUT_DIR / "predictions.csv", ",", OUTPUT_DIR / "evaluation.csv")

In [ ]:
# 8) 요약
from collections import Counter
scored = [e for e in eval_rows if not e.get("error")]
def acc(rows):
    return f"{sum(e['correct'] for e in rows)}/{len(rows)} = {100*sum(e['correct'] for e in rows)/len(rows):.2f}%" if rows else "-"

summary = {
    "env": ENV,
    "images_run": len(dumped),
    "errors": [e["image_id"] for e in eval_rows if e.get("error")],
    "accuracy_all": acc(scored),
    "accuracy_approved": acc([e for e in scored if e["truth_source"] == "approved"]),
    "accuracy_candidate": acc([e for e in scored if e["truth_source"] == "candidate"]),
    "accuracy_existing_300": acc([e for e in scored if "existing_300" in e["label_sources"]]),
    "accuracy_new_401": acc([e for e in scored if e["label_sources"] == "incoming_432"]),
    "methods": dict(Counter(e["method"] for e in scored)),
    "wrong_first_guess": dict(Counter(e["first_guess"] for e in scored if not e["correct"])),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=1))
print("\nClaude에게 보낼 파일:", OUTPUT_DIR / "ocr_dump.jsonl", ",", OUTPUT_DIR / "evaluation.csv", ",", OUTPUT_DIR / "summary.json")